# Сделки с жилыми помещениями в Казани: очистка данных

В этом ноутбуке сырой набор преобразуется в очищенную master-таблицу. Очистка консервативная: исправляются типы и формат значений, но реальные договоры, ДДУ, многообъектные сделки, пропуски и необычные наблюдения не удаляются без достаточного основания.

In [66]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_FILE = DATA_DIR / "kazan_residential_transactions_2025_raw.csv"
OUTPUT_FILE = DATA_DIR / "kazan_residential_transactions_2025_clean.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "Сначала выполните ноутбук 01_data_preparation.ipynb. "
        f"Ожидаемый файл: {INPUT_FILE}"
    )

raw = pd.read_csv(INPUT_FILE, dtype="string", encoding="utf-8")
print(f"Исходный размер: {raw.shape}")
raw.head()

Исходный размер: (5584, 18)


,number,okato,region_code,district,city,quarter_cad_number,street,realestate_type_code,wall_material_code,year_build,floor,purpose_code,area,period_start_date,deal_price,currency,doc_type,quarter
0,1,92401380000,16,<NA>,Казань,16:50:160205,Хусаина Мавлютова,002001003000,061001001001,1986,9,206002000000,56.9,2025-01-01,8900000.0,рубль,ДКП,Q1
1,3,92401367000,16,<NA>,Казань,16:50:011102,Старая,002001003000,<NA>,<NA>,<NA>,206002000000,7991.3099999999995,2025-01-01,54923916,рубль,ДДУ,Q1
2,1,92401367000,16,<NA>,Казань,16:50:011705,Николая Столбова,002001003000,061001003000,2020,2,206002000000,62.1,2025-01-01,26400000,рубль,ДКП,Q1
3,1,92401385000,16,<NA>,Казань,16:50:060510,Комарова,002001003000,061001007001,1969,5,206002000000,59.4,2025-01-01,9000000.0,рубль,ДКП,Q1
4,1,92401363000,16,<NA>,Казань,16:50:220526,Молодежная,002001003000,061001001001,1982,5,206002000000,12.9,2025-01-01,2200000.0,рубль,ДКП,Q1


## Проверка границ набора

Перед очисткой повторно проверяются город, вид объекта, назначение, валюта и тип договора. Ошибка на этом этапе означает, что входной файл сформирован неверно.

In [67]:
scope_checks = pd.DataFrame(
    {
        "проверка": [
            "ОКАТО за пределами Казани",
            "неверный код региона",
            "неверный тип объекта",
            "неверное назначение помещения",
            "неожиданная валюта",
            "неожиданный тип договора",
        ],
        "строк": [
            (~raw["okato"].str.startswith("92401", na=False)).sum(),
            (raw["region_code"] != "16").sum(),
            (raw["realestate_type_code"] != "002001003000").sum(),
            (raw["purpose_code"] != "206002000000").sum(),
            (raw["currency"] != "рубль").sum(),
            (~raw["doc_type"].isin(["ДКП", "ДДУ"])).sum(),
        ],
    }
)

if scope_checks["строк"].sum() > 0:
    raise ValueError("Входной файл не прошел проверку границ набора")

scope_checks

,проверка,строк
0,ОКАТО за пределами Казани,0
1,неверный код региона,0
2,неверный тип объекта,0
3,неверное назначение помещения,0
4,неожиданная валюта,0
5,неожиданный тип договора,0


## Названия и типы данных

`number` переименовывается в `property_count`, чтобы отразить его смысл: это количество объектов в договоре. Числовые поля и дата преобразуются в подходящие типы. Исходное текстовое значение этажа сохраняется в `floor_raw`, поскольку в трех строках вместо номера указаны осмысленные категории.

In [68]:
clean = raw.copy()
clean = clean.rename(columns={"number": "property_count"})
clean["floor_raw"] = clean["floor"]

clean["property_count"] = pd.to_numeric(
    clean["property_count"], errors="coerce"
).astype("Int64")
clean["year_build"] = pd.to_numeric(
    clean["year_build"], errors="coerce"
).astype("Int64")
clean["floor"] = pd.to_numeric(
    clean["floor"], errors="coerce"
).astype("Int64")
clean["area"] = pd.to_numeric(clean["area"], errors="coerce")
clean["deal_price"] = pd.to_numeric(
    clean["deal_price"], errors="coerce"
)
clean["period_start_date"] = pd.to_datetime(
    clean["period_start_date"], errors="coerce"
)

clean.dtypes

property_count                   Int64
okato                   string[python]
region_code             string[python]
district                string[python]
city                    string[python]
quarter_cad_number      string[python]
street                  string[python]
realestate_type_code    string[python]
wall_material_code      string[python]
year_build                       Int64
floor                            Int64
purpose_code            string[python]
area                           Float64
period_start_date       datetime64[ns]
deal_price                     Float64
currency                string[python]
doc_type                string[python]
quarter                 string[python]
floor_raw               string[python]
dtype: object

## Текстовые значения

У используемых текстовых полей удаляются пробелы по краям, а пустые строки заменяются пропусками. Пропуски не заполняются вымышленными значениями.

In [69]:
text_columns = [
    "district",
    "street",
    "wall_material_code",
    "currency",
    "doc_type",
]

for column in text_columns:
    clean[column] = clean[column].str.strip()
    clean.loc[clean[column] == "", column] = pd.NA
clean[text_columns].head()

,district,street,wall_material_code,currency,doc_type
0,<NA>,Хусаина Мавлютова,061001001001,рубль,ДКП
1,<NA>,Старая,<NA>,рубль,ДДУ
2,<NA>,Николая Столбова,061001003000,рубль,ДКП
3,<NA>,Комарова,061001007001,рубль,ДКП
4,<NA>,Молодежная,061001001001,рубль,ДКП


## Удаление избыточных столбцов

После проверки удаляются `region_code`, `city` и `currency`. Код региона и валюта постоянны, а принадлежность к Казани уже подтверждена более надежным кодом ОКАТО. Эти столбцы не различают наблюдения и не помогут анализу или модели.

In [70]:
redundant_columns = ["region_code", "city", "currency"]
clean = clean.drop(columns=redundant_columns)
clean.columns

Index(['property_count', 'okato', 'district', 'quarter_cad_number', 'street',
       'realestate_type_code', 'wall_material_code', 'year_build', 'floor',
       'purpose_code', 'area', 'period_start_date', 'deal_price', 'doc_type',
       'quarter', 'floor_raw'],
      dtype='object')

## Административный район

Исходный почти пустой столбец `district` заменяется районом, напрямую указанным кодом ОКАТО. Для общего кода Казани `92401000000` район неизвестен и остается пропуском. В результате район определен для 4 501 строки (80,6%), а для 1 083 строк (19,4%) отсутствует. Это ограничивает районный анализ: пропуски могут быть распределены по районам неслучайно, поэтому результаты нельзя считать полным описанием всего рынка Казани.

In [71]:
district_by_okato = {
    "92401363000": "Авиастроительный",
    "92401367000": "Вахитовский",
    "92401370000": "Кировский",
    "92401377000": "Московский",
    "92401379000": "Ново-Савиновский",
    "92401380000": "Приволжский",
    "92401385000": "Советский",
}

clean["district"] = clean["okato"].map(district_by_okato)
clean["district"].value_counts(dropna=False)

district
Советский           1439
NaN                 1083
Приволжский          935
Ново-Савиновский     628
Кировский            559
Вахитовский          352
Авиастроительный     338
Московский           250
Name: count, dtype: int64

## Дубликаты и базовая валидность

Удаляются только полные дубликаты. Похожие строки сохраняются, потому что после обезличивания они могут относиться к разным договорам. Договор без положительного количества объектов или цены считается критической ошибкой; пропущенная площадь допускается и отмечается отдельно.

In [72]:
rows_before_deduplication = len(clean)
clean = clean.drop_duplicates().copy()
duplicates_removed = rows_before_deduplication - len(clean)

invalid_core_row = (
    clean["property_count"].isna()
    | (clean["property_count"] <= 0)
    | clean["deal_price"].isna()
    | (clean["deal_price"] <= 0)
    | clean["period_start_date"].isna()
)

if invalid_core_row.sum() > 0:
    raise ValueError("Обнаружены строки с невалидными ключевыми полями договора")

print(f"Удалено полных дубликатов: {duplicates_removed}")
print(f"Договоров с пропущенной площадью: {clean['area'].isna().sum()}")

Удалено полных дубликатов: 0
Договоров с пропущенной площадью: 6


## Аналитические признаки

Договоры разделяются на сделки с одним и несколькими объектами. Цена квадратного метра рассчитывается только для одного объекта: в многообъектном договоре неизвестны цены отдельных помещений. Широкие пороги отмечают значения для проверки, но не удаляют их.

In [73]:
clean["is_single_property_contract"] = clean["property_count"] == 1
clean["is_multi_property_contract"] = clean["property_count"] > 1
clean["contract_segment"] = "один объект"
clean.loc[
    clean["is_multi_property_contract"],
    "contract_segment",
] = "несколько объектов"

valid_individual_price = (
    clean["is_single_property_contract"]
    & clean["area"].notna()
    & (clean["area"] > 0)
)
clean["price_per_sqm"] = pd.NA
clean.loc[valid_individual_price, "price_per_sqm"] = (
    clean.loc[valid_individual_price, "deal_price"]
    / clean.loc[valid_individual_price, "area"]
)
clean["price_per_sqm"] = pd.to_numeric(
    clean["price_per_sqm"], errors="coerce"
)

clean["needs_price_review"] = (
    clean["is_single_property_contract"]
    & (
        (clean["area"] < 10)
        | (clean["area"] > 300)
        | (clean["deal_price"] < 500_000)
        | (clean["deal_price"] > 100_000_000)
        | (clean["price_per_sqm"] < 30_000)
        | (clean["price_per_sqm"] > 500_000)
    )
)
clean["needs_price_review"] = clean["needs_price_review"].fillna(False)
clean["needs_price_review"] = clean["needs_price_review"].astype(bool)
clean["has_complete_area_data"] = clean["area"].notna()
clean["is_area_missing"] = clean["area"].isna()

clean[["contract_segment", "price_per_sqm", "needs_price_review"]].head()

,contract_segment,price_per_sqm,needs_price_review
0,один объект,156414.762742,False
1,несколько объектов,NaN,False
2,один объект,425120.772947,False
3,один объект,151515.151515,False
4,один объект,170542.635659,False


## Проверка результата

Контрольная таблица подтверждает, что сохранены оба сегмента. `area` ниже означает записанную в источнике площадь договора; интерпретацию площади многообъектных договоров необходимо уточнить по документации Росреестра.

In [74]:
segment_groups = clean.groupby("contract_segment")
segment_summary = segment_groups.agg(
    договоров=("contract_segment", "size"),
    объектов_недвижимости=("property_count", "sum"),
    записанная_площадь_кв_м=("area", "sum"),
    общая_стоимость_договоров=("deal_price", "sum"),
    договоров_без_площади=("is_area_missing", "sum"),
)

print(f"Очищенный размер: {clean.shape}")
print(f"Всего объектов в договорах: {clean['property_count'].sum():,.0f}")
print(f"Строк с ценой квадратного метра: {clean['price_per_sqm'].notna().sum():,}")
print(f"Строк для проверки цены: {clean['needs_price_review'].sum():,}")
segment_summary

Очищенный размер: (5584, 23)
Всего объектов в договорах: 16,693
Строк с ценой квадратного метра: 5,020
Строк для проверки цены: 146


,договоров,объектов_недвижимости,записанная_площадь_кв_м,общая_стоимость_договоров,договоров_без_площади
contract_segment,,,,,
несколько объектов,562,11671,54309914.973,139654229457.860016,4
один объект,5022,5022,250062.24,38559162614.540001,2


## Сохранение

Очищенная master-таблица сохраняется отдельно. Все дальнейшие ноутбуки должны читать этот файл, а не изменять исходный набор.

In [75]:
clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"Очищенный файл сохранен: {OUTPUT_FILE}")

Очищенный файл сохранен: c:\Users\Artem\Documents\Kazan_Housing\Kazan-housing\data\kazan_residential_transactions_2025_clean.csv
